In [1]:
from mylib.ml.dbgen import *
from mylib.ml.utils import deform_and_rattle
from mylib.materials_project import get_material
from ase.io import read, write
from ase.visualize import view
from ase import Atoms, Atom
from pathlib import Path
import pandas as pd
from pymatgen.io.ase import AseAtomsAdaptor
import numpy as np
from ase.build import bulk

/home/chris/anaconda3/lib/python3.12/site-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))


In [3]:
api_key = "re1OnovEumCUghRMnWUJafOubEYo7fo1"

In [25]:
el = get_material("mp-7000", api_key)
atoms = AseAtomsAdaptor.get_atoms(el.structure)
# el_dir = Path("/home/chris/Desktop/PHD/Zn")
# el_dir.mkdir()

# write(el_dir/"POSCAR", atoms)

# write("sio2.extxyz", atoms)

In [26]:
view(atoms)

<Popen: returncode: None args: ['/home/chris/anaconda3/bin/python', '-m', 'a...>

In [56]:

def add_vacuum(atoms, vaccum=12):
    tmp = atoms.copy()
    tmp.cell += np.diag([0, 0, 2*vaccum])
    tmp.center()
    return tmp

In [57]:
def add_Cu(atoms_original, buffer: float = 1.0, max_nCuGa = 3):
    atoms = atoms_original.copy()
    rng = np.random.default_rng()
    cell = np.array(atoms.cell)
    new_atoms = atoms.copy()
    
    nCuGa = rng.integers(low=1, high=max_nCuGa+1)
    for i in range(nCuGa):
        pt = rng.choice(["Cu", "Ga"])
        u = rng.random(2)
        if u.sum() > 1:
            u = 1 - u
        max_z = np.max(new_atoms.positions[:, 2])
        
        height = rng.uniform(max_z + 1.1, max_z + 2)
        pos = np.sum(cell[:2] * u[:, None], axis=0)
        pos[2] = height
        new_atoms.append(Atom(pt, position=pos))
    
    distances = new_atoms.get_all_distances()
    np.fill_diagonal(distances, np.inf)
    if np.any(distances < buffer):
        # return add_Cu(atoms, buffer=buffer)
        print("Error")
        # print(max_z, height, cell)
    return new_atoms

In [62]:
new = [add_vacuum(atom) for atom in deform_and_rattle(atoms, num=100, max_deform=0, max_rattle=0.15)]
final = [add_Cu(i, buffer=0.8) for i in new]

In [59]:
len(final)

1000

In [60]:
final

[MSONAtoms(symbols='Si3O6Cu2', pbc=True, cell=[[2.510751, -4.348749, 0.0], [2.510751, 4.348749, 0.0], [0.0, 0.0, 29.51057]]),
 MSONAtoms(symbols='Si3O6Cu', pbc=True, cell=[[2.510751, -4.348749, 0.0], [2.510751, 4.348749, 0.0], [0.0, 0.0, 29.51057]]),
 MSONAtoms(symbols='Si3O6Ga', pbc=True, cell=[[2.510751, -4.348749, 0.0], [2.510751, 4.348749, 0.0], [0.0, 0.0, 29.51057]]),
 MSONAtoms(symbols='Si3O6CuGaCu', pbc=True, cell=[[2.510751, -4.348749, 0.0], [2.510751, 4.348749, 0.0], [0.0, 0.0, 29.51057]]),
 MSONAtoms(symbols='Si3O6Ga', pbc=True, cell=[[2.510751, -4.348749, 0.0], [2.510751, 4.348749, 0.0], [0.0, 0.0, 29.51057]]),
 MSONAtoms(symbols='Si3O6GaCu', pbc=True, cell=[[2.510751, -4.348749, 0.0], [2.510751, 4.348749, 0.0], [0.0, 0.0, 29.51057]]),
 MSONAtoms(symbols='Si3O6Cu', pbc=True, cell=[[2.510751, -4.348749, 0.0], [2.510751, 4.348749, 0.0], [0.0, 0.0, 29.51057]]),
 MSONAtoms(symbols='Si3O6CuGa', pbc=True, cell=[[2.510751, -4.348749, 0.0], [2.510751, 4.348749, 0.0], [0.0, 0.0, 29.5

In [63]:
write("../val_substarte.extxyz", final)

In [37]:
write("si_oxide_vacuum.extxyz", new[:180])
write("val_si_oxide_vacuum.extxyz", new[180:])

In [69]:
cell = np.array(test.cell)
u = np.random.random(size=2)
if u.sum() > 1:
    u = 1 - u
np.sum(cell[:2]*u[:, None], axis = 0)

array([ 1.37043683, -2.37238829,  0.        ])

In [ ]:
def append_molecule_on_cluster(atoms:Atoms, molecule:Atoms, min_distance:float, max_distance:float):
    COM = atoms.get_center_of_mass()
    

In [2]:
dftdf = pd.read_csv("/home/chris/Desktop/PHD/apotelesmata/DATA/dft.csv")
surfacedf = pd.read_csv("/home/chris/Desktop/PHD/apotelesmata/DATA/surface_foo.csv")
surfacedf["miller"] = surfacedf["miller"].astype(str)

In [3]:
ga_concentration = np.linspace(0, 0.5, 5)
structures = ["fcc", "bcc", "hcp"]
surface_structures = [
    "fcc100",
    "fcc110",
    "fcc111",
    "fcc111_root",
    "bcc100",
    "bcc110",
    "bcc111",
    "bcc111_root",
    "hcp0001",
    "hcp0001_root",
    "hcp10m10"
]

In [4]:
total_atoms = []
for cs in structures:
    for c in ga_concentration:

        total_atoms.extend(
            deform_and_rattle(build_cluster(cs, dftdf, c, "Cu", "Ga", repeat=9), surface=False)
        )

In [ ]:
total_atoms = []
for cs in structures:
    for c in ga_concentration:

        total_atoms.extend(
            deform_and_rattle(build_bulk_structure(cs, dftdf, c, "Cu", "Ga"))
        )


write("bulk.extxyz", total_atoms)

In [4]:
total_atoms = []
for cs in surface_structures:
    for c in ga_concentration:

        total_atoms.extend(
            deform_and_rattle(build_surface(cs, dftdf, c, "Cu", "Ga"), surface=True)
        )



In [ ]:
def append_O_on_surface(surface_org:Atoms, nO=1):
    surface = surface_org.copy()
    rng = np.random.default_rng()
    max_height = np.max(surface.positions, axis = 0)[2]
    
    cell = np.array(surface.cell)
    for _ in range(nO):
        u = rng.uniform(0.0, 1.0, size=2)
        if u.sum() > 1:
            u = 1 - u
        pos = np.sum(cell[:2]*u[:, None], axis = 0)
        height = max_height + rng.uniform(1.8, 2.2)
        pos[2] = height
        
        surface.append(Atom("O", position=pos))
        
    return surface
    
    

In [16]:

oxygenated_surfaces = [append_O_on_surface(surface) for surface in total_atoms]
oxygenated_surfaces.extend([append_O_on_surface(surface, nO=2) for surface in total_atoms])

In [18]:
validation = [append_O_on_surface(surface) for surface in total_atoms]

In [19]:
write("../../O_surface.extxyz", oxygenated_surfaces)
write("../../val_O_surface.extxyz", validation[:100])

/home/chris/anaconda3/lib/python3.12/site-packages/ase/io/extxyz.py:311: UserWarning: Skipping unhashable information adsorbate_info
  warnings.warn('Skipping unhashable information '


In [ ]:
write("surfaces.extxyz",total_atoms)

In [5]:
write("clusters.extxyz",total_atoms)

In [ ]:
total_atoms = []
for c in ga_concentration:
    tmp = create_nnpt("fcc", "Cu", "Ga", c, dftdf, surfacedf, vacuum=10, N = 256)
    total_atoms.extend(deform_and_rattle(tmp, surface=False, max_deform=0))

In [ ]:
write("nanopts.extxyz",total_atoms)